# Hapus Kolom SHAPE.AREA pada Shapefile

Notebook ini membaca seluruh atribut data dari file `.shp`, menampilkan daftar kolom, lalu menghapus kolom `SHAPE.AREA` (dan `SHAPE.LEN` apabila ada) dan menyimpan hasilnya ke file baru.

## 1. Install dependency

Jalankan sel ini hanya bila `geopandas` belum terinstall.

In [ ]:
%pip install geopandas

## 2. Import library & definisikan path

In [2]:
import geopandas as gpd
import os

# Path folder yang berisi subfolder kabupaten, masing-masing berisi file .shp
input_dir = "Batas Administrasi"

# Folder output untuk hasil
output_dir = "shp_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Folder input : {input_dir}")
print(f"Folder output: {output_dir}")

Folder input : Batas Administrasi
Folder output: shp_output


## 3. Daftar semua file .shp (rekursif)

Semua file `.shp` di dalam `Batas Administrasi` (termasuk subfolder kabupaten) dikumpulkan beserta nama kabupatennya.

In [3]:
shp_files = sorted(
    os.path.join(root, f)
    for root, _, files in os.walk(input_dir)
    for f in files
    if f.lower().endswith(".shp")
)
print(f"Ditemukan {len(shp_files)} file .shp:")
for p in shp_files:
    print(f"  - {p}")

Ditemukan 3 file .shp:
  - Batas Administrasi/Kabupaten Biak Numfor/Kab_Biak_Numfor.shp
  - Batas Administrasi/Kabupaten Merauke/Kab_Merauke.shp
  - Batas Administrasi/Kabupaten Mimika/Kab_Mimika.shp


## 4. Baca seluruh atribut & hapus kolom SHAPE.AREA

Setiap shapefile dibaca, atribut lengkapnya ditampilkan, lalu kolom `SHAPE.AREA` (dan `SHAPE.LEN`) dihapus. Hasil disimpan sebagai shapefile baru di folder output.

In [4]:
removed_cols = ["SHAPE.AREA", "SHAPE.LEN"]

for path in shp_files:
    shp = os.path.basename(path)
    gdf = gpd.read_file(path)

    print("=" * 60)
    print(f"File : {shp}")
    print(f"CRS  : {gdf.crs} | Jumlah fitur: {len(gdf)}")
    print("KOLOM SEBELUM :")
    print(list(gdf.columns))

    # Cari kolom yang benar-benar ada
    drop = [c for c in removed_cols if c in gdf.columns]
    if drop:
        gdf = gdf.drop(columns=drop)
        print(f"Dibuang kolom: {drop}")
    else:
        print("Tidak ada kolom SHAPE.AREA / SHAPE.LEN yang ditemukan.")

    print("KOLOM SESUDAH :")
    print(list(gdf.columns))

    # Simpan dengan nama kabupaten sebagai prefix (untuk menghindari bentrok nama)
    out_path = os.path.join(output_dir, shp)
    gdf.to_file(out_path)
    print(f"Tersimpan -> {out_path}")
    print()

File : Kab_Biak_Numfor.shp
CRS  : EPSG:4326 | Jumlah fitur: 1
KOLOM SEBELUM :
['fid', 'OBJECTID', 'NAMOBJ', 'FCODE', 'REMARK', 'METADATA', 'SRS_ID', 'KDBBPS', 'KDCBPS', 'KDCPUM', 'KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM', 'WADMKC', 'WADMKD', 'WADMKK', 'WADMPR', 'WIADKC', 'WIADKK', 'WIADPR', 'WIADKD', 'SHAPE.AREA', 'SHAPE.LEN', 'geometry']
Dibuang kolom: ['SHAPE.AREA', 'SHAPE.LEN']
KOLOM SESUDAH :
['fid', 'OBJECTID', 'NAMOBJ', 'FCODE', 'REMARK', 'METADATA', 'SRS_ID', 'KDBBPS', 'KDCBPS', 'KDCPUM', 'KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM', 'WADMKC', 'WADMKD', 'WADMKK', 'WADMPR', 'WIADKC', 'WIADKK', 'WIADPR', 'WIADKD', 'geometry']
Tersimpan -> shp_output/Kab_Biak_Numfor.shp

File : Kab_Merauke.shp
CRS  : EPSG:4326 | Jumlah fitur: 1
KOLOM SEBELUM :
['fid', 'OBJECTID', 'NAMOBJ', 'FCODE', 'REMARK', 'METADATA', 'SRS_ID', 'KDBBPS', 'KDCBPS', 'KDCPUM', 'KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM', 'WADMKC', 'WADMKD', 

## 5. Verifikasi hasil

Baca ulang file output untuk memastikan kolom `SHAPE.AREA` sudah hilang dan data atribut lain tetap utuh.

In [5]:
for shp in sorted(os.listdir(output_dir)):
    if shp.lower().endswith(".shp"):
        gdf = gpd.read_file(os.path.join(output_dir, shp))
        print(f"{shp} -> {len(gdf)} fitur | kolom: {list(gdf.columns)}")
        print("  Contoh baris:")
        print(gdf.drop(columns="geometry").head(3).to_string(index=False))
        print()

Kab_Biak_Numfor.shp -> 1 fitur | kolom: ['fid', 'OBJECTID', 'NAMOBJ', 'FCODE', 'REMARK', 'METADATA', 'SRS_ID', 'KDBBPS', 'KDCBPS', 'KDCPUM', 'KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM', 'WADMKC', 'WADMKD', 'WADMKK', 'WADMPR', 'WIADKC', 'WIADKK', 'WIADPR', 'WIADKD', 'geometry']
  Contoh baris:
 fid  OBJECTID      NAMOBJ      FCODE REMARK                   METADATA SRS_ID KDBBPS KDCBPS KDCPUM KDEBPS KDEPUM KDPBPS KDPKAB KDPPUM      LUASWH  TIPADM WADMKC WADMKD      WADMKK WADMPR WIADKC WIADKK WIADPR  WIADKD
 1.0      1136 Biak Numfor BA03050040   None TASWIL5000020230907KABKOTA   4326   None   None   None   None   None   None  91.06     91 2257.778573       4   None   None Biak Numfor  Papua   None   None   None       0

Kab_Merauke.shp -> 1 fitur | kolom: ['fid', 'OBJECTID', 'NAMOBJ', 'FCODE', 'REMARK', 'METADATA', 'SRS_ID', 'KDBBPS', 'KDCBPS', 'KDCPUM', 'KDEBPS', 'KDEPUM', 'KDPBPS', 'KDPKAB', 'KDPPUM', 'LUASWH', 'TIPADM', 'WADMKC', 'WADMKD', 'WADMKK', 'WADMPR',

## Catatan

- Kolom `SHAPE.AREA` adalah field otomatis dari ArcGIS/QGIS (vendor *.dbf* module). Nilai luasnya tidak ter-update saat geometri berubah, sehingga aman dihapus.
- Jika ingin menghitung ulang luas, tambahkan kolom `luas` dengan `gdf.geometry.area`.
- Simpan dalam format **GeoPackage (\.gpkg)** apabila jumlah kolom > 255 atau nama kolom > 10 karakter (batasan Shapefile).